In [1]:
import pandas as pd
import numpy as np
import ta
import os

SYMBOLS = ["AAPL", "MSFT", "NVDA", "JPM", "BAC", "TSLA", "AMZN"]

In [ ]:
def compute_features(df):
    close  = df["Close"]
    high   = df["High"]
    low    = df["Low"]

    # === Indicators ===
    df["sma_20"]  = ta.trend.SMAIndicator(close, 20).sma_indicator()
    df["sma_200"] = ta.trend.SMAIndicator(close, 200).sma_indicator()

    df["rsi_14"] = ta.momentum.RSIIndicator(close, 14).rsi()
    macd = ta.trend.MACD(close)
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()

    bb = ta.volatility.BollingerBands(close, 20)
    df["bb_upper"] = bb.bollinger_hband()
    df["bb_lower"] = bb.bollinger_lband()
    df["bb_width"] = bb.bollinger_wband()

    # === Derived features ===
    df["rsi_norm"] = df["rsi_14"] / 100
    df["close_over_sma20"] = close / df["sma_20"]
    df["close_over_sma200"] = close / df["sma_200"]

    bb_range = (df["bb_upper"] - df["bb_lower"]).replace(0, np.nan)
    df["bb_position"] = (close - df["bb_lower"]) / bb_range

    # === Returns / Volatility ===
    df["ret_1"] = close.pct_change(1)
    df["ret_5"] = close.pct_change(5)
    df["ret_10"] = close.pct_change(10)

    df["vol_5"] = df["ret_1"].rolling(5).std()
    df["vol_20"] = df["ret_1"].rolling(20).std()

    # === Volume ===
    df["vol_change"] = df["Volume"].pct_change()
    df["vol_ma"] = df["Volume"].rolling(10).mean()
    df["vol_ratio"] = df["Volume"] / df["vol_ma"]

    # === Price position ===
    df["high_20"] = high.rolling(20).max()
    df["low_20"] = low.rolling(20).min()
    df["price_position"] = (close - df["low_20"]) / (df["high_20"] - df["low_20"])

    # === Advanced ===
    df["trend_strength"] = df["sma_20"] / df["sma_200"]
    df["volatility_ratio"] = df["vol_5"] / df["vol_20"]

    # === SHIFT (lag-1 so all indicators are from previous day) ===
    features_to_shift = [
        "sma_20","sma_200",
        "rsi_14","macd","macd_signal",
        "bb_upper","bb_lower","bb_width",
        "rsi_norm","close_over_sma20","close_over_sma200","bb_position",
        "ret_1","ret_5","ret_10",
        "vol_5","vol_20",
        "vol_change","vol_ma","vol_ratio",
        "high_20","low_20","price_position",
        "trend_strength","volatility_ratio"
    ]

    df[features_to_shift] = df[features_to_shift].shift(1)

    # === Target: 5-day forward return > 1% ===
    future = close.shift(-5)
    df["target"] = ((future / close - 1) > 0.01).astype(int)

    return df.dropna()

In [3]:
import pandas as pd

all_features = []

for symbol in SYMBOLS:
    df = pd.read_csv(f"../data/raw/{symbol}.csv", parse_dates=["Date"])
    
    df["symbol"] = symbol
    df = compute_features(df)
    
    all_features.append(df)

full_features = pd.concat(all_features)

# save ไฟล์เดียว
full_features.to_csv(
    "../data/processed/features/features_all.csv",
    index=False
)

print("✓ dataset ready:", full_features.shape)

✓ dataset ready: (16212, 33)
